# DeepFish marine detector — YOLOv8n (Kaggle T4)

Replaces the freshwater-trained detector with one trained on **DeepFish** (YOLO-Fish
benchmark, single-class "fish"), and proves the domain-gap fix with a **before/after** on the
held-out benchmark TEST split. Runs on Kaggle T4 — not locally.

- Data: DeepFish YOLO export (public GDrive) using the **benchmark-official split** (no re-split).
- Backgrounds: NOAA "Labeled Fishes in the Wild" negatives as hard-negatives in TRAIN only (~<=10%).
- Label format was verified locally (no conversion): `outputs/label_check.png`.

In [ ]:
# B1 — setup. CRITICAL: do NOT let pip replace Kaggle's GPU-matched torch/torchvision.
# A naive `pip install ultralytics` pulls a torch wheel whose CUDA arch may not match the
# assigned GPU -> "CUDA error: no kernel image is available for execution on the device".
# Pin the pre-installed torch/torchvision in a constraints file so the resolver can't swap them.
import glob
import os
import random
import shutil
import subprocess
import sys
from pathlib import Path

import torch
import torchvision
import yaml

Path("constraints.txt").write_text(
    f"torch=={torch.__version__}\ntorchvision=={torchvision.__version__}\n"
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-c", "constraints.txt", "ultralytics", "albumentations", "gdown"],
    check=True,
)

# GPU diagnostic — confirm torch STILL matches the assigned GPU after the install.
print("torch              :", torch.__version__)
print("torch.version.cuda :", torch.version.cuda)
print("cuda available     :", torch.cuda.is_available())
print("device name        :", torch.cuda.get_device_name(0))
print("device capability  :", torch.cuda.get_device_capability(0))
print("arch list          :", torch.cuda.get_arch_list())

from ultralytics import YOLO

SEED = 42
random.seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("device", DEVICE)

WORK = Path("/kaggle/working")
RAW = WORK / "deepfish_raw"
DATASET = WORK / "deepfish"          # assembled YOLO dataset root
RAW.mkdir(parents=True, exist_ok=True)
for sub in ["images/train", "images/val", "images/test",
            "labels/train", "labels/val", "labels/test"]:
    (DATASET / sub).mkdir(parents=True, exist_ok=True)

In [ ]:
# A1 — fetch the DeepFish YOLO export (YOLO-Fish benchmark, public GDrive). Images stay on Kaggle.
import gdown

FULL_ID = "10Pr4lLeSGTfkjA40ReGSC8H3a9onfMZ0"   # annotated DeepFish dataset (all images + labels)
TEST_ID = "1iHPFbqo-B2iVytusHic9s8VuMlLIMu5-"   # official DeepFish TEST set (909 imgs / 3029 boxes)

full_zip = RAW / "deepfish_full.zip"
test_zip = RAW / "deepfish_test.zip"
if not full_zip.exists():
    gdown.download(id=FULL_ID, output=str(full_zip), quiet=False)
if not test_zip.exists():
    gdown.download(id=TEST_ID, output=str(test_zip), quiet=False)
# If gdown hits a Google quota error, mirror the file to a Kaggle Dataset, attach it, and point
# full_zip / test_zip at the /kaggle/input/... path instead.

(RAW / "full").mkdir(exist_ok=True)
(RAW / "test").mkdir(exist_ok=True)
shutil.unpack_archive(str(full_zip), str(RAW / "full"))
shutil.unpack_archive(str(test_zip), str(RAW / "test"))
print("unpacked full + test exports")

In [ ]:
# A1 — assemble the YOLO dataset, CLIP-DISJOINT. DeepFish frames are consecutive video stills
# named "<clip>_f######"; neighbouring frames are near-identical, so a frame-level split leaks
# the test clips into train. Here a whole clip lives in exactly ONE split.
#   TEST  = the official benchmark test zip (kept as-is for comparability).
#   TRAIN/VAL = full export MINUS every frame whose clip appears in TEST, then split by clip.
import re
from collections import defaultdict

def pairs(root):
    """stem -> (jpg_path, txt_path or None), recursive, regardless of internal folder layout."""
    out = {}
    for jpg in glob.glob(str(Path(root) / "**" / "*.jpg"), recursive=True):
        txt = str(Path(jpg).with_suffix(".txt"))
        out[Path(jpg).stem] = (jpg, txt if os.path.exists(txt) else None)
    return out

def clip_of(stem):
    """Clip/habitat id = filename with the trailing frame index '_f######' stripped."""
    return re.sub(r"_f\d+$", "", stem)

test_pairs = pairs(RAW / "test")
full_pairs = pairs(RAW / "full")
test_clips = {clip_of(s) for s in test_pairs}

# Train/val pool: only frames whose clip is NOT in TEST -> clip-disjoint from the test set.
pool_stems = [s for s in full_pairs if clip_of(s) not in test_clips]
pool_by_clip = defaultdict(list)
for s in pool_stems:
    pool_by_clip[clip_of(s)].append(s)

pool_clips = sorted(pool_by_clip)
random.Random(SEED).shuffle(pool_clips)
n_val_clips = max(1, round(0.12 * len(pool_clips)))    # ~12% of CLIPS (not frames) to val
val_clips = set(pool_clips[:n_val_clips])
tr_clips = set(pool_clips[n_val_clips:])
tr_stems = [s for c in tr_clips for s in pool_by_clip[c]]
val_stems = [s for c in val_clips for s in pool_by_clip[c]]

def place(stem, src_pairs, split):
    jpg, txt = src_pairs[stem]
    shutil.copy(jpg, DATASET / "images" / split / Path(jpg).name)
    dst_txt = DATASET / "labels" / split / (Path(jpg).stem + ".txt")
    if txt:
        shutil.copy(txt, dst_txt)
    else:
        dst_txt.write_text("")   # image with no boxes -> background

for s in tr_stems:
    place(s, full_pairs, "train")
for s in val_stems:
    place(s, full_pairs, "val")
for s in test_pairs:
    place(s, test_pairs, "test")

# HARD GATE: no clip may appear on two sides.
tr_clip_set = {clip_of(s) for s in tr_stems}
val_clip_set = {clip_of(s) for s in val_stems}
assert tr_clip_set.isdisjoint(test_clips), f"LEAK: train∩test clips {tr_clip_set & test_clips}"
assert val_clip_set.isdisjoint(test_clips), f"LEAK: val∩test clips {val_clip_set & test_clips}"
assert tr_clip_set.isdisjoint(val_clip_set), f"LEAK: train∩val clips {tr_clip_set & val_clip_set}"

def split_counts(split):
    imgs = glob.glob(str(DATASET / "images" / split / "*.jpg"))
    boxes = empt = 0
    clips = set()
    for im in imgs:
        clips.add(clip_of(Path(im).stem))
        lp = DATASET / "labels" / split / (Path(im).stem + ".txt")
        lines = [ln for ln in lp.read_text().splitlines() if ln.strip()] if lp.exists() else []
        boxes += len(lines)
        empt += not lines
    return len(imgs), boxes, empt, len(clips)

print("split = CLIP-DISJOINT (official TEST kept; train/val split by clip)")
for sp in ["train", "val", "test"]:
    n, b, e, c = split_counts(sp)
    print(f"{sp:5s}: {n:5d} imgs | {b:6d} boxes | {e} empty-label | {c} clips")
print(f"clip overlap train∩test={len(tr_clip_set & test_clips)} "
      f"val∩test={len(val_clip_set & test_clips)} train∩val={len(tr_clip_set & val_clip_set)}")

In [ ]:
# A2 — NOAA "Labeled Fishes in the Wild" negatives as hard-negative backgrounds (TRAIN only).
# Cap at <=10% of total images; report the actual background fraction.
NOAA_URL = ("https://storage.googleapis.com/nmfs_odp_swfsc/"
            "Fisheries%20Resources%20Division/Labeled_Fishes_In_The_Wild.zip")
noaa_zip = RAW / "noaa.zip"
if not noaa_zip.exists():
    subprocess.run(["wget", "-q", "-O", str(noaa_zip), NOAA_URL], check=True)
(RAW / "noaa").mkdir(exist_ok=True)
shutil.unpack_archive(str(noaa_zip), str(RAW / "noaa"))

# Locate the negative (no-fish) seabed images inside the archive.
all_imgs = [p for p in glob.glob(str(RAW / "noaa" / "**" / "*.*"), recursive=True)
            if p.lower().endswith((".jpg", ".jpeg", ".png"))]
neg = [p for p in all_imgs if "negativ" in p.lower()]
if not neg:
    print("WARNING: no path matched 'negativ'. Archive image dirs:")
    dirs = sorted({str(Path(p).parent) for p in all_imgs})
    for d in dirs[:20]:
        print("  ", d)
print(f"NOAA negative images found: {len(neg)}")

n_tr = len(glob.glob(str(DATASET / "images" / "train" / "*.jpg")))
n_va = len(glob.glob(str(DATASET / "images" / "val" / "*.jpg")))
n_te = len(glob.glob(str(DATASET / "images" / "test" / "*.jpg")))
total_fish = n_tr + n_va + n_te
cap = int(0.10 * total_fish)
use = neg[: max(0, min(len(neg), cap))]
for i, p in enumerate(use):
    name = f"noaa_neg_{i:04d}.jpg"
    shutil.copy(p, DATASET / "images" / "train" / name)
    (DATASET / "labels" / "train" / f"noaa_neg_{i:04d}.txt").write_text("")  # empty = background

total_all = total_fish + len(use)
frac = 100 * len(use) / max(1, total_all)
print(f"added {len(use)} NOAA backgrounds to TRAIN (cap {cap} = 10% of {total_fish} fish images)")
print(f"background fraction: {len(use)}/{total_all} = {frac:.1f}% of all images")

In [ ]:
# A3 — data.yaml (single class). Benchmark TEST stays held out for the final number.
data_yaml = {
    "path": str(DATASET),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 1,
    "names": ["fish"],
}
yaml_path = WORK / "deepfish.yaml"
yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False))
print(yaml_path.read_text())

In [ ]:
# B2 — BEFORE: current freshwater detector on the held-out DeepFish TEST set.
# yolo_fish.pt is attached via a private Kaggle dataset; find it wherever it got mounted
# under /kaggle/input (folder name varies if the dataset is renamed/forked).
_fw = sorted(glob.glob("/kaggle/input/**/yolo_fish.pt", recursive=True))
assert _fw, (
    "freshwater weights not found: no yolo_fish.pt under /kaggle/input — "
    "attach the freshwater-fish-detector dataset"
)
FRESHWATER_PT = _fw[0]
print("freshwater weights:", FRESHWATER_PT)

before = YOLO(FRESHWATER_PT).val(
    data=str(yaml_path), split="test", imgsz=640, device=DEVICE,
    project=str(WORK / "runs"), name="before_freshwater", verbose=False,
)
before_map50, before_map = before.box.map50, before.box.map
print(f"BEFORE (freshwater) on DeepFish TEST: mAP50={before_map50:.4f}  mAP50-95={before_map:.4f}")
print("Expect this to be poor — that poorness IS the domain-gap evidence.")

In [ ]:
# B3 — TRAIN from COCO-pretrained yolov8n (NOT the freshwater checkpoint -> no freshwater bias).
model = YOLO("yolov8n.pt")
train_res = model.train(
    data=str(yaml_path),
    imgsz=640,
    epochs=100,
    patience=20,
    batch=-1,        # auto-batch for the T4
    amp=True,
    seed=SEED,
    device=DEVICE,
    cache=False,     # do NOT cache to RAM — multi-GB image set OOMs Kaggle's ~13 GB
    hsv_s=0.9,       # raise saturation/value jitter to fight the blue/green water cast
    hsv_v=0.6,
    project=str(WORK / "runs"),
    name="deepfish_yolov8n",
)
best_pt = Path(train_res.save_dir) / "weights" / "best.pt"
print("best.pt:", best_pt)

In [ ]:
# B4 — AFTER: eval new best.pt on carved val and held-out benchmark TEST; NOAA background FP count.
new = YOLO(str(best_pt))
val_m = new.val(data=str(yaml_path), split="val", imgsz=640, device=DEVICE,
                project=str(WORK / "runs"), name="after_val", verbose=False)
test_m = new.val(data=str(yaml_path), split="test", imgsz=640, device=DEVICE,
                 project=str(WORK / "runs"), name="after_test", verbose=False)

def row(m):
    return m.box.map50, m.box.map, m.box.mp, m.box.mr

v = row(val_m)
t = row(test_m)
print(f"AFTER  val : mAP50={v[0]:.4f} mAP50-95={v[1]:.4f} P={v[2]:.4f} R={v[3]:.4f}")
print(f"AFTER  TEST: mAP50={t[0]:.4f} mAP50-95={t[1]:.4f} P={t[2]:.4f} R={t[3]:.4f}")

# Background false positives: predict on the NOAA negatives (should be ~0 — what hard-negatives buy).
bg_imgs = glob.glob(str(DATASET / "images" / "train" / "noaa_neg_*.jpg"))
fp = 0
if bg_imgs:
    for r in new.predict(bg_imgs, imgsz=640, device=DEVICE, conf=0.25, verbose=False):
        fp += len(r.boxes)
    print(f"NOAA background frames: {len(bg_imgs)} | total false-positive boxes: {fp}")
else:
    print("No NOAA background frames present to test.")

In [ ]:
# B5 — RESULTS. Headline: same held-out DeepFish TEST set, before vs after.
print("============== BEFORE / AFTER (DeepFish TEST, held out) ==============")
print(f'{"detector":26s} {"mAP50":>8s} {"mAP50-95":>9s}')
print(f'{"freshwater (before)":26s} {before_map50:8.4f} {before_map:9.4f}')
print(f'{"DeepFish yolov8n (after)":26s} {t[0]:8.4f} {t[1]:9.4f}')
print(f"delta mAP50: {t[0] - before_map50:+.4f}")
print()
print("External sanity check (BALLPARK ONLY): YOLO-Fish published DeepFish AP ~= 0.76.")
print(f"  our mAP50 on TEST = {t[0]:.4f}")
print("  near/above with yolov8n = solid; far below = pipeline bug, not a real result.")
print("  NOT a benchmark-beat claim — model and AP/IoU definitions differ.")

In [ ]:
# B6 — export new detector + keep run artifacts.
OUT = WORK / "export"
OUT.mkdir(exist_ok=True)
shutil.copy(best_pt, OUT / "yolo_fish.pt")
print("best.pt copied to", OUT / "yolo_fish.pt")
print("run artifacts (PR_curve.png, confusion_matrix.png, results.png) in:", best_pt.parent.parent)

## Download
1. Output tab → `/kaggle/working/export/yolo_fish.pt` → download → place locally at `weights/yolo_fish.pt`.
2. Also grab `runs/deepfish_yolov8n/{PR_curve.png, confusion_matrix.png, results.png}` for the writeup.
3. Local real-time picks it up automatically (`realtime.py --detector` default = `weights/yolo_fish.pt`).